# NB1 — Data EDA + Provenance Log

Structural econometrics pipeline, Phase 1. Authors the provenance log, panel fingerprint, and exploratory diagnostics consumed by NB2 and NB3.

**Status:** skeleton (Task 1c). Cells are authored progressively in later Phase 1 tasks.

> **Gate Verdict:** populated after NB2 and NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary into this cell.

## 1. Setup and Data Availability Statement

### Manifest

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* (Social Science Data Editors README template).

**Why used.** The manifest is the first component of the Data Availability Statement (DAS) block: it enumerates every raw source feeding the DuckDB warehouse, together with its row count, observed date range, SHA-256 of the downloaded bytes, and the URL or filesystem path from which it was retrieved.

**Relevance to our results.** Every downstream estimate produced by this notebook chain traces back to a named source in this table. Readers and reviewers can audit whether a specific β̂ or test statistic depends on a source whose fetch is reproducible, whose hash is recorded, and whose row count matches what the pipeline ingested.

**Connection to simulator.** Layer 2 consumers — in particular the RAN payoff bootstrap — re-materialise the DuckDB from the same raw sources when regenerating fitted parameters. The manifest is the contract they read: a source that cannot be re-downloaded from the URL-or-path recorded here cannot be used to refit the simulator's parameters.


In [ ]:
import importlib.util
import sys
from pathlib import Path

import duckdb
import pandas as pd

# Load env.py by file path (it is not on sys.path).
_ENV_PATH = Path.cwd() / "env.py"
if not _ENV_PATH.exists():
    # Fallback: notebook may be executed from a different cwd
    _ENV_PATH = (
        Path(__file__).resolve().parent / "env.py"
        if "__file__" in globals()
        else _ENV_PATH
    )

_spec = importlib.util.spec_from_file_location("fx_vol_env", _ENV_PATH)
env = importlib.util.module_from_spec(_spec)
# Register in sys.modules BEFORE exec so that `from __future__ import
# annotations` + `@dataclass(slots=True)` name resolution works.
sys.modules["fx_vol_env"] = env
_spec.loader.exec_module(env)

# Import the query API by file path (scripts/ is not on sys.path).
_API_PATH = env._CONTRACTS_DIR / "scripts" / "econ_query_api.py"
_api_spec = importlib.util.spec_from_file_location("econ_query_api", _API_PATH)
econ_query_api = importlib.util.module_from_spec(_api_spec)
# Same sys.modules registration needed for dataclass(slots=True) resolution.
sys.modules["econ_query_api"] = econ_query_api
_api_spec.loader.exec_module(econ_query_api)

# Open a read-only connection to the structural-econ DuckDB. Kept alive for
# subsequent §1 trios.
conn = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Fetch the manifest and render as a DataFrame for display.
manifest_rows = econ_query_api.get_manifest(conn)
manifest_df = pd.DataFrame(
    [
        {
            "source": r.source,
            "row_count": r.row_count,
            "date_min": r.date_min,
            "date_max": r.date_max,
            "sha256": (r.sha256[:12] + "…") if r.sha256 else None,
            "url_or_path": r.url_or_path,
            "status": r.status,
        }
        for r in manifest_rows
    ]
)
manifest_df


The manifest above enumerates every raw source currently materialised in the DuckDB warehouse. For each source the table reports:

- **`source`** — canonical short name used as a foreign key by downstream tables.
- **`row_count`** — number of records ingested from that source into its primary landing table.
- **`date_min` / `date_max`** — observed date range of the ingested rows (data coverage, not download timestamp).
- **`sha256`** — truncated hash of the downloaded bytes (first 12 characters shown); the full 64-character digest lives in the `download_manifest` table.
- **`url_or_path`** — public URL or filesystem path from which the source was retrieved, so a fresh clone can reproduce the download.
- **`status`** — provenance lifecycle marker (e.g. `ok`, `imputed`, `failed`).

Any row with a null `sha256` or null `url_or_path` indicates a provenance gap — the pipeline recorded the source but the audit trail is incomplete for that fetch. Such rows are flagged here so readers can decide whether a given downstream computation inherits a provenance gap.
